# PKG community map: real data (graph ML workshop demo)

Draws the Payment Knowledge Graph (PKG) at community level, so the workshop can see what the bank's own payment network looks like.

**Scope.** Customer-to-customer flows from `pkg_edge_monthly` between organisations (party type O on both ends, the B2B subgraph), self-transfers excluded, one month by default. Counterparties stay out: at median degree about 1 they add size, not structure.

**Parts.**
* **Part A (Spark):** extract the in-scope edges and node attributes, save them as parquet under `OUT_DIR`.
* **Part B (graph and pictures):** runs in any kernel with pandas and NetworkX. It reads the parquet, removes hubs, finds communities (cuGraph Leiden when a GPU is there, else igraph Leiden, else NetworkX Louvain) and writes the pictures. To run Part B on the GPU box: run the Config and Setup cells there with `RUN_PART_A = False`, then the Part B cells.

**Outputs** (all under `OUT_DIR`): `community_map.html` / `.png`, `hairball_vs_dehubbed.png`, `drill_<community>.html` / `.png`, `community_composition.html` / `.png`, `community_profile.csv`, `membership.parquet`, `hubs_removed.csv`.

**Governance.** This is real customer data. Present it from your own screen and don't hand out the HTML or PNG files. `SHOW_NAMES = False` keeps customer names out of every picture and table.

In [ ]:
# Config
# ---- inputs (PKG pipeline tables) ----
DB          = "bdahd01p_dlcdi1_cdi_tm"
EDGE_TABLE  = f"{DB}.pkg_edge_monthly"   # src_id, src_kind, dst_id, dst_kind, is_self, amount, n_txn, month
CUST_DIM    = f"{DB}.pkg_cust_dim"       # mdm_id, customer_name, party_type, is_tm, naics2, naics_desc, naics_status
MONTHS      = ["2026-08"]                # one month; list three months for a quarter view
NODE_SCOPE  = "org"                      # "org": party_type O on both ends (B2B); "tm": is_tm on both ends
RUN_PART_A  = True                       # False in a kernel without Spark: Part B reads the parquet in OUT_DIR
MAX_COLLECT_PAIRS = 15_000_000           # fail fast instead of pulling too much to the driver

# ---- graph ----
MIN_PAIR_AMOUNT = 0.0        # drop customer pairs below this many dollars over the period
HUB_RULE        = "pct"      # "pct": degree OR strength above the HUB_PCT quantile (ladder rung P99_9)
                             # "median_mult": degree above HUB_MEDIAN_MULT x median degree
HUB_PCT         = 0.999
HUB_MEDIAN_MULT = 15
RESOLUTION      = 1.0        # higher = more, smaller communities
SEED            = 42

# ---- pictures ----
TOP_K_COMMUNITIES  = 60      # communities drawn on the map (largest first)
MIN_COMMUNITY_SIZE = 10
LINKS_PER_COMMUNITY = 3      # strongest links kept per community on the map
DRILL_COMMUNITIES  = None    # None = the two largest; or a list such as ["C003", "C011"]
DRILL_MAX_NODES    = 250
HAIRBALL_NODES     = 1200
SHOW_NAMES         = False   # keep False when presenting
OUT_DIR            = "../metrics/community/workshop_map"

In [ ]:
# Setup: environment shims, quiet logs, Spark session (Part A only)
import os, time, math, warnings
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

import numpy as np
# Cloudera SPARK3 PySpark predates NumPy 2 and still calls these aliases in toPandas
for _alias, _target in (("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)):
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)

import pandas as pd
# PySpark 3.3 createDataFrame calls DataFrame.iteritems, removed in pandas 2.0
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

import networkx as nx
from IPython.display import display, HTML
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs(OUT_DIR, exist_ok=True)

spark = None
if RUN_PART_A:
    from pyspark.sql import SparkSession, functions as F
    spark = (SparkSession.builder.appName("pkg_community_map")
             .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
             .config("spark.executor.memory", "4g")
             .config("spark.driver.memory", "4g")
             .config("spark.driver.maxResultSize", "4g")
             .config("spark.executor.memoryOverhead", "2g")
             .config("spark.dynamicAllocation.maxExecutors", "512")
             .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
             .config("spark.sql.execution.arrow.pyspark.enabled", "true")
             .config("spark.ui.showConsoleProgress", "false")
             .enableHiveSupport()
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    try:   # the cluster's Atlas lineage hook logs a harmless error on every table read/write
        _L = spark._jvm.org.apache.logging.log4j
        for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
                   "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
                   "org.apache.spark.network", "io.netty"):
            _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
    except Exception:
        pass

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
OTHER = "#a3a29b"
NAICS2 = {"11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
          "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing", "42": "Wholesale",
          "44": "Retail", "45": "Retail", "48": "Transportation", "49": "Transportation",
          "51": "Information", "52": "Finance and insurance", "53": "Real estate",
          "54": "Professional services", "55": "Holding companies", "56": "Admin and support",
          "61": "Education", "62": "Health care", "71": "Arts and recreation",
          "72": "Food services", "81": "Other services", "92": "Public administration"}


def show(title, items):
    """One consolidated HTML block per cell (outputs are shared as screenshots)."""
    parts = [f"<h4 style='margin:6px 0'>{title}</h4>"]
    for it in items:
        if isinstance(it, pd.DataFrame):
            parts.append(it.to_html(index=False, border=0, justify="left",
                                    float_format=lambda x: f"{x:,.3f}"))
        else:
            parts.append(f"<div style='margin:2px 0'>{it}</div>")
    display(HTML("".join(parts)))


def usd(x):
    x = float(x)
    for unit, d in (("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if abs(x) >= d:
            return f"USD {x / d:,.1f}{unit}"
    return f"USD {x:,.0f}"


show("Setup", [f"pandas {pd.__version__}, numpy {np.__version__}, networkx {nx.__version__}",
               f"Spark {spark.version if spark else 'not started (RUN_PART_A = False)'}",
               f"OUT_DIR {os.path.abspath(OUT_DIR)}"])

## Part A (Spark): extract the B2B customer-to-customer edges

One row per directed customer pair over `MONTHS`, amounts summed (cast to double so pandas gets floats, not Decimals). Both ends must be in scope. Node attributes come from `pkg_cust_dim`.

In [ ]:
# A1. Extract in-scope edges and node attributes, save as parquet
assert spark is not None, "Part A needs RUN_PART_A = True and a Spark kernel"
t0 = time.time()
months_sql = ", ".join(f"'{m}'" for m in MONTHS)

c2c = spark.sql(f"""
    SELECT CAST(src_id AS STRING) AS src, CAST(dst_id AS STRING) AS dst,
           SUM(CAST(amount AS DOUBLE)) AS amount, SUM(CAST(n_txn AS BIGINT)) AS n_txn
    FROM {EDGE_TABLE}
    WHERE month IN ({months_sql})
      AND src_kind = 'CUST' AND dst_kind = 'CUST'
      AND COALESCE(is_self, FALSE) = FALSE
      AND src_id <> dst_id
    GROUP BY CAST(src_id AS STRING), CAST(dst_id AS STRING)
""")
dim = spark.sql(f"""
    SELECT CAST(mdm_id AS STRING) AS node, customer_name, party_type,
           CAST(is_tm AS BOOLEAN) AS is_tm, CAST(naics2 AS STRING) AS naics2,
           naics_desc, naics_status
    FROM {CUST_DIM}
""")
if NODE_SCOPE == "org":
    scope = dim.filter(F.col("party_type") == "O")
elif NODE_SCOPE == "tm":
    scope = dim.filter(F.col("is_tm"))
else:
    raise ValueError("NODE_SCOPE must be 'org' or 'tm'")
scope_ids = scope.select("node")

in_scope = (c2c.join(scope_ids.withColumnRenamed("node", "src"), "src", "left_semi")
               .join(scope_ids.withColumnRenamed("node", "dst"), "dst", "left_semi")).cache()
tot = c2c.agg(F.count(F.lit(1)).alias("pairs"), F.sum("amount").alias("amount")).collect()[0]
kept = in_scope.agg(F.count(F.lit(1)).alias("pairs"), F.sum("amount").alias("amount")).collect()[0]
assert kept["pairs"] and kept["pairs"] > 0, f"No in-scope pairs for {MONTHS} / {NODE_SCOPE}"
assert kept["pairs"] <= MAX_COLLECT_PAIRS, (
    f"{kept['pairs']:,} pairs exceeds MAX_COLLECT_PAIRS; use fewer months or raise MIN_PAIR_AMOUNT")

edges_pd = in_scope.select("src", "dst", "amount", "n_txn").toPandas()
node_ids = (in_scope.select(F.col("src").alias("node"))
            .union(in_scope.select(F.col("dst").alias("node"))).distinct())
nodes_pd = node_ids.join(dim, "node", "left").toPandas()
assert nodes_pd.node.is_unique, "pkg_cust_dim should hold one row per mdm_id"
in_scope.unpersist()


def _save(df, name):
    try:
        df.to_parquet(os.path.join(OUT_DIR, f"{name}.parquet"), index=False)
        return f"{name}.parquet"
    except ImportError:          # no pyarrow: fall back to CSV
        df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
        return f"{name}.csv"


saved = [_save(edges_pd, "edges"), _save(nodes_pd, "nodes")]
valid = (nodes_pd.naics_status == "valid").mean()
show("A1. Extract", [
    f"Months {', '.join(MONTHS)} &middot; scope <b>{NODE_SCOPE}</b>",
    f"All customer-to-customer pairs: {tot['pairs']:,} &middot; {usd(tot['amount'] or 0)}",
    f"In scope: <b>{kept['pairs']:,}</b> pairs ({kept['pairs'] / tot['pairs']:.1%}) &middot; "
    f"{usd(kept['amount'])} ({kept['amount'] / tot['amount']:.1%} of dollars)",
    f"Nodes {len(nodes_pd):,} &middot; TM {nodes_pd.is_tm.fillna(False).astype(bool).mean():.1%} &middot; valid NAICS {valid:.1%}",
    f"Saved {', '.join(saved)} &middot; {time.time() - t0:,.0f}s",
])

## Part B: graph, hubs, communities, pictures

From here on nothing needs Spark. On the GPU box, run Config and Setup with `RUN_PART_A = False`, then continue here.

In [ ]:
# B1. Load, build the undirected weighted graph, remove hubs
t0 = time.time()


def _read(name):
    p = os.path.join(OUT_DIR, name)
    if os.path.exists(p + ".parquet"):
        return pd.read_parquet(p + ".parquet")
    return pd.read_csv(p + ".csv", dtype={"src": str, "dst": str, "node": str})


edges = _read("edges")
nodes = _read("nodes").set_index("node")
nodes["sector"] = [NAICS2.get(str(c)[:2]) if s == "valid" and pd.notna(c) else None
                   for c, s in zip(nodes.naics2, nodes.naics_status)]

e = edges[edges.amount > MIN_PAIR_AMOUNT]
a, b = e.src.to_numpy(dtype=object), e.dst.to_numpy(dtype=object)
lo = np.where(a < b, a, b)
hi = np.where(a < b, b, a)
pairs = (pd.DataFrame({"u": lo, "v": hi, "amount": e.amount.to_numpy(), "n_txn": e.n_txn.to_numpy()})
         .groupby(["u", "v"], as_index=False).sum())

ends = pd.concat([pairs[["u", "amount"]].rename(columns={"u": "node"}),
                  pairs[["v", "amount"]].rename(columns={"v": "node"})])
deg = ends.groupby("node").size()
strength = ends.groupby("node").amount.sum()
if HUB_RULE == "pct":
    is_hub = (deg > deg.quantile(HUB_PCT)) | (strength > strength.quantile(HUB_PCT))
    rule = f"degree or strength above P{HUB_PCT * 100:g}"
elif HUB_RULE == "median_mult":
    is_hub = deg > HUB_MEDIAN_MULT * deg.median()
    rule = f"degree above {HUB_MEDIAN_MULT} x median ({deg.median():.0f})"
else:
    raise ValueError("HUB_RULE must be 'pct' or 'median_mult'")
hubs = set(deg.index[is_hub.to_numpy()])

hub_tab = pd.DataFrame({"node": list(hubs)})
hub_tab["degree"] = hub_tab.node.map(deg)
hub_tab["strength"] = hub_tab.node.map(strength)
hub_tab["name"] = hub_tab.node.map(nodes.customer_name)
hub_tab["sector"] = hub_tab.node.map(nodes.sector)
hub_tab["naics_desc"] = hub_tab.node.map(nodes.naics_desc)
hub_tab = hub_tab.sort_values("degree", ascending=False)
hub_tab.to_csv(os.path.join(OUT_DIR, "hubs_removed.csv"), index=False)

keep = ~pairs.u.isin(hubs) & ~pairs.v.isin(hubs)
pairs_dh = pairs[keep].reset_index(drop=True)
codes, uniq = pd.factorize(np.concatenate([pairs_dh.u.to_numpy(), pairs_dh.v.to_numpy()]))
cu, cv = codes[:len(pairs_dh)], codes[len(pairs_dh):]
n = len(uniq)
n_nodes_dh = n
dh_deg = np.bincount(cu, minlength=n) + np.bincount(cv, minlength=n)

view = hub_tab.head(15).copy()
view["strength"] = view.strength.map(usd)
view["label"] = view["name"] if SHOW_NAMES else [f"Hub {i + 1}" for i in range(len(view))]
show("B1. Graph and hubs", [
    f"Undirected pairs {len(pairs):,} &middot; nodes {deg.size:,} &middot; {usd(pairs.amount.sum())}",
    f"Degree median {deg.median():.0f}, P99 {deg.quantile(.99):.0f}, max {deg.max():,}",
    f"Hub rule: {rule} &rarr; <b>{len(hubs):,}</b> hubs removed (list in hubs_removed.csv)",
    f"Left after hub removal: {len(pairs_dh):,} pairs ({keep.mean():.1%}), {n_nodes_dh:,} nodes, "
    f"{usd(pairs_dh.amount.sum())} ({pairs_dh.amount.sum() / pairs.amount.sum():.1%} of dollars)",
    view[["label", "sector", "naics_desc", "degree", "strength"]],
    f"{time.time() - t0:,.0f}s",
])

## Communities

Weight = log(1 + dollars) per customer pair, so a few very large flows don't decide the partition. The engine is picked automatically: cuGraph Leiden on a GPU, else igraph Leiden, else NetworkX Louvain on CPU (slower on a big graph; it prints how long it took).

In [ ]:
# B2. Find communities
t0 = time.time()
w = np.log1p(pairs_dh.amount.to_numpy())
notes = []


def detect(cu, cv, w, n):
    try:
        import cudf, cugraph
        gdf = cudf.DataFrame({"src": cu.astype("int32"), "dst": cv.astype("int32"),
                              "weight": w.astype("float32")})
        G = cugraph.Graph(directed=False)
        G.from_cudf_edgelist(gdf, source="src", destination="dst", edge_attr="weight", renumber=True)
        try:
            parts, _ = cugraph.leiden(G, max_iter=100, resolution=RESOLUTION, random_state=SEED)
        except TypeError:          # older cuGraph has no random_state
            parts, _ = cugraph.leiden(G, max_iter=100, resolution=RESOLUTION)
        parts = parts.to_pandas()
        memb = np.full(n, -1)
        memb[parts["vertex"].to_numpy()] = parts["partition"].to_numpy()
        return memb, "cuGraph Leiden (GPU)"
    except Exception as ex:
        notes.append(f"cuGraph not used ({type(ex).__name__})")
    try:
        import igraph as ig, leidenalg
        g = ig.Graph(n=n, edges=np.column_stack([cu, cv]).tolist())
        part = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition, weights=w.tolist(),
                                        resolution_parameter=RESOLUTION, seed=SEED, n_iterations=-1)
        return np.asarray(part.membership), "igraph Leiden (CPU)"
    except Exception as ex:
        notes.append(f"igraph/leidenalg not used ({type(ex).__name__})")
    G = nx.Graph()
    G.add_weighted_edges_from(zip(cu.tolist(), cv.tolist(), w.tolist()))
    comms = nx.community.louvain_communities(G, weight="weight", resolution=RESOLUTION, seed=SEED)
    memb = np.full(n, -1)
    for i, c in enumerate(comms):
        memb[list(c)] = i
    return memb, "NetworkX Louvain (CPU)"


raw, engine = detect(cu, cv, w, n)
assert (raw >= 0).all(), "every node should get a community"
# renumber by size: C001 is the largest
sizes = pd.Series(raw).value_counts()
rank = {c: i for i, c in enumerate(sizes.index)}
memb = np.array([rank[c] for c in raw])
K = memb.max() + 1


def modularity(cu, cv, w, memb, K):
    m = w.sum()
    s = np.bincount(cu, w, len(memb)) + np.bincount(cv, w, len(memb))
    same = memb[cu] == memb[cv]
    L = np.bincount(memb[cu][same], w[same], K)
    D = np.bincount(memb, s, K)
    return float((L / m - (D / (2 * m)) ** 2).sum())


Q = modularity(cu, cv, w, memb, K)
cid = np.array([f"C{i + 1:03d}" for i in range(K)])
membership = pd.DataFrame({"node": uniq, "community": cid[memb]})
try:
    membership.to_parquet(os.path.join(OUT_DIR, "membership.parquet"), index=False)
except ImportError:
    membership.to_csv(os.path.join(OUT_DIR, "membership.csv"), index=False)

sz = sizes.to_numpy()
inside = (memb[cu] == memb[cv])
show("B2. Communities", [
    f"Engine <b>{engine}</b> &middot; resolution {RESOLUTION} &middot; {time.time() - t0:,.0f}s"
    + (f" &middot; {'; '.join(notes)}" if notes else ""),
    f"Communities {K:,} &middot; modularity <b>{Q:.3f}</b> (0 = no structure; above about 0.3 = clear structure)",
    f"Size: largest {sz[0]:,}, {(sz >= MIN_COMMUNITY_SIZE).sum():,} with {MIN_COMMUNITY_SIZE}+ members, "
    f"{(sz <= 3).sum():,} tiny ones (2-3 members)",
    f"Top {TOP_K_COMMUNITIES} communities hold {sz[:TOP_K_COMMUNITIES].sum() / n:.1%} of nodes",
    f"Dollars inside communities: {pairs_dh.amount.to_numpy()[inside].sum() / pairs_dh.amount.sum():.1%}",
])

In [ ]:
# B3. Hairball vs de-hubbed: the same businesses with and without the hubs, coloured by community
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import breadth_first_order

rng = np.random.default_rng(SEED)
A = coo_matrix((np.ones(len(cu)), (cu, cv)), shape=(n, n)).tocsr()
A = A + A.T
comm_of = dict(zip(uniq, memb))

sample = []
seeds = np.where((dh_deg >= 5) & (dh_deg <= 60))[0]
seeds = seeds if len(seeds) >= 3 else np.argsort(-dh_deg)[:3]
for s in rng.choice(seeds, 3, replace=False):
    order = breadth_first_order(A, s, directed=False, return_predecessors=False)
    sample.extend(order[:HAIRBALL_NODES // 3].tolist())
S = set(uniq[np.unique(sample)])
top_hubs = hub_tab.node.head(6).tolist()

sub_dh = pairs_dh[pairs_dh.u.isin(S) & pairs_dh.v.isin(S)]
sub_h = pairs[(pairs.u.isin(top_hubs) & pairs.v.isin(S)) | (pairs.v.isin(top_hubs) & pairs.u.isin(S))]
G_with = nx.from_pandas_edgelist(pd.concat([sub_dh, sub_h]), "u", "v")
G_wo = nx.from_pandas_edgelist(sub_dh, "u", "v")

fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="#fcfcfb")
for ax, G, title in ((axes[0], G_with, f"With the {len(top_hubs)} largest hubs"),
                     (axes[1], G_wo, "Same businesses, hubs removed (colour = community)")):
    pos = nx.spring_layout(G, seed=SEED, iterations=60)
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.3, alpha=0.35, edge_color="#8d8c85")
    hub_nodes = [x for x in G if x in set(top_hubs)]
    rest = [x for x in G if x not in set(top_hubs)]
    big = pd.Series([comm_of[x] for x in rest]).value_counts().index[:len(PALETTE)].tolist()
    col = [PALETTE[big.index(comm_of[x])] if comm_of[x] in big else OTHER for x in rest]
    nx.draw_networkx_nodes(G, pos, nodelist=rest, ax=ax, node_size=7, node_color=col, linewidths=0)
    if hub_nodes:
        nx.draw_networkx_nodes(G, pos, nodelist=hub_nodes, ax=ax, node_size=160, node_color="#eb6834", linewidths=0)
    ax.set_title(title, fontsize=15, color="#0b0b0b")
    ax.axis("off")
fig.tight_layout()
hair_png = os.path.join(OUT_DIR, "hairball_vs_dehubbed.png")
fig.savefig(hair_png, dpi=150, facecolor=fig.get_facecolor())
plt.close(fig)
show("B3. Hairball vs de-hubbed", [
    f"Sample: {len(S):,} businesses (three breadth-first snowballs), {G_wo.number_of_edges():,} links among them; "
    f"the hubs add {len(sub_h):,} more links",
    f"Saved {hair_png}",
])

In [ ]:
# B4. Community profile: size, dollars, dominant sector and purity
node_df = pd.DataFrame({"node": uniq, "c": memb, "degree": dh_deg})
node_df["strength"] = (np.bincount(cu, pairs_dh.amount.to_numpy(), n)
                       + np.bincount(cv, pairs_dh.amount.to_numpy(), n))
node_df["sector"] = node_df.node.map(nodes.sector)
node_df["is_tm"] = node_df.node.map(nodes.is_tm).fillna(False).astype(bool)

amt = pairs_dh.amount.to_numpy()
internal = pd.Series(amt[inside]).groupby(memb[cu][inside]).sum()
cross = ~inside
external = (pd.Series(np.concatenate([amt[cross], amt[cross]]))
            .groupby(np.concatenate([memb[cu][cross], memb[cv][cross]])).sum())

prof = node_df.groupby("c").agg(members=("node", "size"), tm_members=("is_tm", "sum"),
                                 with_sector=("sector", "count"))
prof["internal_usd"] = internal.reindex(prof.index).fillna(0)
prof["external_usd"] = external.reindex(prof.index).fillna(0)
prof["inside_share"] = prof.internal_usd / (prof.internal_usd + prof.external_usd).replace(0, np.nan)

sec_counts = node_df.dropna(subset=["sector"]).groupby(["c", "sector"]).size()
top_sec = sec_counts.groupby(level=0, group_keys=False).apply(lambda s: s.sort_values(ascending=False))
dom = top_sec.groupby(level=0).head(1).reset_index().set_index("c")
prof["dominant_sector"] = dom.sector.reindex(prof.index)
prof["purity"] = (dom[0] / prof.with_sector).reindex(prof.index)
prof["top_sectors"] = top_sec.groupby(level=0).apply(
    lambda s: ", ".join(f"{k[1]} {v / s.sum():.0%}" for k, v in s.head(3).items())).reindex(prof.index)
prof.insert(0, "community", cid[prof.index])
prof["label"] = [f"{c} {d}" if isinstance(d, str) else c for c, d in zip(prof.community, prof.dominant_sector)]
prof.to_csv(os.path.join(OUT_DIR, "community_profile.csv"), index=False)

view = prof.head(20).copy()
for col in ("internal_usd", "external_usd"):
    view[col] = view[col].map(usd)
view["inside_share"] = view.inside_share.map(lambda x: f"{x:.0%}" if pd.notna(x) else "")
view["purity"] = view.purity.map(lambda x: f"{x:.0%}" if pd.notna(x) else "")
show("B4. Twenty largest communities", [
    view[["community", "members", "tm_members", "internal_usd", "inside_share",
          "dominant_sector", "purity", "top_sectors"]],
    "Purity = share of members (with a valid NAICS) in the dominant sector. Low purity is the interesting part: "
    "a community that mixes sectors is a supply chain or a regional ecosystem, not an industry code.",
])

In [ ]:
# B5. The community map: one bubble per community, links = dollars flowing between communities
t0 = time.time()
top = prof[prof.members >= MIN_COMMUNITY_SIZE].head(TOP_K_COMMUNITIES)
top_ids = set(top.index)

mc, mv = memb[cu], memb[cv]
sel = (mc != mv) & np.isin(mc, list(top_ids)) & np.isin(mv, list(top_ids))
links = (pd.DataFrame({"a": np.minimum(mc[sel], mv[sel]), "b": np.maximum(mc[sel], mv[sel]), "usd": amt[sel]})
         .groupby(["a", "b"], as_index=False).usd.sum())
links["rank_a"] = links.groupby("a").usd.rank(ascending=False)
links["rank_b"] = links.groupby("b").usd.rank(ascending=False)
links = links[(links.rank_a <= LINKS_PER_COMMUNITY) | (links.rank_b <= LINKS_PER_COMMUNITY)].copy()

QG = nx.Graph()
QG.add_nodes_from(top.index)
lw = np.log1p(links.usd.to_numpy())
for (x, y), wt in zip(links[["a", "b"]].itertuples(index=False, name=None), lw / lw.max()):
    QG.add_edge(x, y, weight=float(wt))
pos = nx.spring_layout(QG, weight="weight", seed=SEED, k=1.6 / math.sqrt(max(len(QG), 1)), iterations=200)

sectors_ranked = top.dominant_sector.value_counts().index.tolist()
color_of = {s: PALETTE[i] for i, s in enumerate(sectors_ranked[:len(PALETTE)])}
top = top.assign(color=[color_of.get(s, OTHER) for s in top.dominant_sector],
                 legend=[s if s in color_of else "Other sectors" for s in top.dominant_sector])
size_px = 14 + 46 * np.sqrt(top.members / top.members.max())
bins = np.quantile(lw, [0, .5, .8, .95, 1.0]) if len(lw) else [0]

# static PNG (for slides)
fig, ax = plt.subplots(figsize=(16, 10), facecolor="#fcfcfb")
for (x, y), l in zip(links[["a", "b"]].itertuples(index=False, name=None), lw):
    ax.plot([pos[x][0], pos[y][0]], [pos[x][1], pos[y][1]], color="#b9b8b0",
            lw=0.5 + 3.5 * (l - lw.min()) / (lw.max() - lw.min() + 1e-9), zorder=1, solid_capstyle="round")
for leg in list(dict.fromkeys(list(color_of) + ["Other sectors"])):
    t = top[top.legend == leg]
    if len(t):
        ax.scatter([pos[i][0] for i in t.index], [pos[i][1] for i in t.index], s=(size_px[t.index] ** 2) / 2.2,
                   color=t.color.iloc[0], edgecolor="#fcfcfb", linewidth=2, zorder=2, label=leg)
for i in top.index[:15]:
    ax.annotate(top.community[i], pos[i], ha="center", va="center", fontsize=8, color="#0b0b0b", zorder=3)
lg = ax.legend(loc="lower left", frameon=False, fontsize=11)
for h in getattr(lg, "legend_handles", getattr(lg, "legendHandles", [])):
    h.set_sizes([140])
ax.set_title(f"PKG community map: {', '.join(MONTHS)}, B2B customer-to-customer, hubs removed "
             f"({len(top)} largest communities)", fontsize=14, color="#0b0b0b", loc="left")
ax.axis("off")
fig.tight_layout()
map_png = os.path.join(OUT_DIR, "community_map.png")
fig.savefig(map_png, dpi=150, facecolor=fig.get_facecolor())
plt.close(fig)

# interactive HTML (plotly.js embedded so it opens offline)
saved = [map_png]
try:
    import plotly.graph_objects as go
    traces = []
    widths = [1, 2, 3.5, 5.5]
    for j in range(len(bins) - 1):
        m = (lw >= bins[j]) & ((lw <= bins[j + 1]) if j == len(bins) - 2 else (lw < bins[j + 1]))
        xs, ys = [], []
        for x, y in links[["a", "b"]].to_numpy()[m]:
            xs += [pos[x][0], pos[y][0], None]
            ys += [pos[x][1], pos[y][1], None]
        traces.append(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=widths[j], color="#c9c8c0"),
                                 hoverinfo="skip", showlegend=False))
    for leg in list(dict.fromkeys(list(color_of) + ["Other sectors"])):
        t = top[top.legend == leg]
        if not len(t):
            continue
        hover = [f"<b>{r.community}</b><br>{r.members:,} members ({r.tm_members:,} TM)<br>"
                 f"inside the community: {usd(r.internal_usd)} ({r.inside_share:.0%})<br>"
                 f"dominant: {r.dominant_sector} ({r.purity:.0%})<br>{r.top_sectors}"
                 for r in t.itertuples()]
        traces.append(go.Scatter(
            x=[pos[i][0] for i in t.index], y=[pos[i][1] for i in t.index], mode="markers+text",
            text=[c if i in top.index[:15] else "" for i, c in zip(t.index, t.community)],
            textposition="middle center", textfont=dict(size=9, color="#0b0b0b"),
            marker=dict(size=size_px[t.index], color=t.color.iloc[0], line=dict(width=2, color="#fcfcfb")),
            name=leg, hovertext=hover, hoverinfo="text"))
    figp = go.Figure(traces)
    figp.update_layout(title=f"PKG community map: {', '.join(MONTHS)} (B2B, hubs removed)",
                       plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", height=820,
                       xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
                       legend=dict(orientation="h", y=-0.03), margin=dict(l=10, r=10, t=50, b=10))
    map_html = os.path.join(OUT_DIR, "community_map.html")
    figp.write_html(map_html, include_plotlyjs=True)
    saved.append(map_html)
except ImportError:
    notes.append("plotly missing: PNG only")
show("B5. Community map", [
    f"{len(top)} communities, {len(links):,} links (each community's {LINKS_PER_COMMUNITY} strongest)",
    f"Colours = dominant sector; bubble area = members; link width = dollars between communities",
    "Saved " + ", ".join(saved), f"{time.time() - t0:,.0f}s",
])

In [ ]:
# B6. Drill-down: node-level view inside the largest communities (names masked)
t0 = time.time()
drill = DRILL_COMMUNITIES or prof.community.head(2).tolist()
saved = []
for c_label in drill:
    c = int(c_label[1:]) - 1
    mem = node_df[node_df.c == c].sort_values("strength", ascending=False).head(DRILL_MAX_NODES)
    ids = set(mem.node)
    sub = pairs_dh[pairs_dh.u.isin(ids) & pairs_dh.v.isin(ids)]
    G = nx.from_pandas_edgelist(sub.assign(weight=np.log1p(sub.amount)), "u", "v", ["weight", "amount"])
    if G.number_of_nodes() == 0:
        continue
    pr = nx.pagerank(G, weight="weight")
    p2 = nx.spring_layout(G, seed=SEED, weight="weight", iterations=100)
    secs = mem.set_index("node").sector.reindex(list(G)).fillna("Unknown sector")
    ranked = secs.value_counts().index.tolist()
    cmap = {s: PALETTE[i] for i, s in enumerate([s for s in ranked if s != "Unknown sector"][:len(PALETTE)])}
    alias = {x: (nodes.customer_name.get(x) if SHOW_NAMES else f"N{i + 1:03d}")
             for i, x in enumerate(mem.node)}
    sizes_n = 5 + 25 * np.sqrt(np.array([pr[x] for x in G]) / max(pr.values()))

    fig, ax = plt.subplots(figsize=(12, 10), facecolor="#fcfcfb")
    nx.draw_networkx_edges(G, p2, ax=ax, width=0.4, alpha=0.4, edge_color="#8d8c85")
    for s in list(cmap) + ["Other"]:
        nl = [x for x in G if (secs[x] == s) or (s == "Other" and secs[x] not in cmap)]
        if nl:
            ax.scatter([p2[x][0] for x in nl], [p2[x][1] for x in nl],
                       s=[(5 + 25 * math.sqrt(pr[x] / max(pr.values()))) ** 2 / 3 for x in nl],
                       color=cmap.get(s, OTHER), edgecolor="#fcfcfb", linewidth=1, label=s, zorder=2)
    lg = ax.legend(frameon=False, fontsize=10, loc="lower left")
    for h in getattr(lg, "legend_handles", getattr(lg, "legendHandles", [])):
        h.set_sizes([90])
    ax.set_title(f"{c_label}: {G.number_of_nodes()} members, {G.number_of_edges()} links "
                 f"(size = PageRank inside the community)", fontsize=13, loc="left")
    ax.axis("off")
    fig.tight_layout()
    png = os.path.join(OUT_DIR, f"drill_{c_label}.png")
    fig.savefig(png, dpi=150, facecolor=fig.get_facecolor())
    plt.close(fig)
    saved.append(png)
    try:
        import plotly.graph_objects as go
        xs, ys = [], []
        for x, y in G.edges():
            xs += [p2[x][0], p2[y][0], None]
            ys += [p2[x][1], p2[y][1], None]
        tr = [go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.6, color="#c9c8c0"),
                         hoverinfo="skip", showlegend=False)]
        for s in list(cmap) + ["Other"]:
            nl = [x for x in G if (secs[x] == s) or (s == "Other" and secs[x] not in cmap)]
            if not nl:
                continue
            tr.append(go.Scatter(
                x=[p2[x][0] for x in nl], y=[p2[x][1] for x in nl], mode="markers", name=s,
                marker=dict(size=[5 + 25 * math.sqrt(pr[x] / max(pr.values())) for x in nl],
                            color=cmap.get(s, OTHER), line=dict(width=1, color="#fcfcfb")),
                hovertext=[f"<b>{alias[x]}</b><br>{secs[x]}<br>{nodes.naics_desc.get(x) or ''}<br>"
                           f"links here: {G.degree(x)}<br>PageRank rank: "
                           f"{sorted(pr, key=pr.get, reverse=True).index(x) + 1}" for x in nl],
                hoverinfo="text"))
        f2 = go.Figure(tr)
        f2.update_layout(title=f"{c_label}: inside one community (names masked)", height=780,
                         plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", xaxis=dict(visible=False),
                         yaxis=dict(visible=False, scaleanchor="x"), margin=dict(l=10, r=10, t=50, b=10))
        h = os.path.join(OUT_DIR, f"drill_{c_label}.html")
        f2.write_html(h, include_plotlyjs=True)
        saved.append(h)
    except ImportError:
        pass
show("B6. Drill-down", [f"Communities: {', '.join(drill)}", "Saved " + ", ".join(saved),
                        f"{time.time() - t0:,.0f}s"])

In [ ]:
# B7. Composition: which sectors make up each of the twenty largest communities
comp_c = prof.head(20)
mix = (node_df[node_df.c.isin(comp_c.index)].dropna(subset=["sector"])
       .groupby(["c", "sector"]).size().unstack(fill_value=0))
mix = mix.div(mix.sum(1), axis=0)
order_s = mix.sum().sort_values(ascending=False).index.tolist()
main = order_s[:len(PALETTE)]
mix["Other sectors"] = mix[[s for s in order_s if s not in main]].sum(1) if len(order_s) > len(main) else 0.0
mix = mix[main + ["Other sectors"]].loc[comp_c.index[::-1]]

fig, ax = plt.subplots(figsize=(14, 9), facecolor="#fcfcfb")
left = np.zeros(len(mix))
for i, s in enumerate(mix.columns):
    ax.barh(range(len(mix)), mix[s], left=left, color=(PALETTE + [OTHER])[i], edgecolor="#fcfcfb",
            linewidth=2, label=s, height=0.8)
    left += mix[s].to_numpy()
ax.set_yticks(range(len(mix)))
ax.set_yticklabels([f"{cid[c]}  ({prof.members[c]:,})" for c in mix.index], fontsize=10)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.legend(ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.06), fontsize=10)
ax.set_title("Sector mix of the twenty largest communities (members with a valid NAICS)", loc="left", fontsize=13)
fig.tight_layout()
comp_png = os.path.join(OUT_DIR, "community_composition.png")
fig.savefig(comp_png, dpi=150, facecolor=fig.get_facecolor())
plt.close(fig)
saved = [comp_png]
try:
    import plotly.graph_objects as go
    f3 = go.Figure([go.Bar(y=[cid[c] for c in mix.index], x=mix[s], name=s, orientation="h",
                           marker=dict(color=(PALETTE + [OTHER])[i], line=dict(color="#fcfcfb", width=2)),
                           hovertemplate="%{y}: " + s + " %{x:.0%}<extra></extra>")
                    for i, s in enumerate(mix.columns)])
    f3.update_layout(barmode="stack", height=700, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
                     xaxis=dict(tickformat=".0%", range=[0, 1]), title="Sector mix of the largest communities",
                     legend=dict(orientation="h", y=-0.08))
    h = os.path.join(OUT_DIR, "community_composition.html")
    f3.write_html(h, include_plotlyjs=True)
    saved.append(h)
except ImportError:
    pass
mixed = (prof.head(20).purity < 0.5).sum()
show("B7. Composition", [f"{mixed} of the 20 largest communities have no sector above 50% of members: "
                         "they are ecosystems (supply chains, regional clusters), not industries.",
                         "Saved " + ", ".join(saved)])

In [ ]:
# B8. What was written
files = sorted(os.listdir(OUT_DIR))
listing = pd.DataFrame({"file": files,
                        "size_kb": [round(os.path.getsize(os.path.join(OUT_DIR, f)) / 1024) for f in files]})
show("B8. Outputs", [listing, "Open community_map.html in a browser for the live demo. "
                     "Present from your own screen; do not share the files."])